# Step 4: Positions

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table, train the toddler and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("4.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.
> - Near the end, a **playground** trains the toddler on your own sentences, with and without positions.

<!-- /notebook-only -->

**Goal:** give each vector a sense of *where* its token sits in the text, and train the toddler again to see what it gains.

In [ ]:
# Run this first (practice only): imports, the tokenizer (Step 1) and the table with our hint rows (Step 2)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.embeddings import COLUMNS, hint_table
from toddler.exercises import check, solution


def short(x):
    return "0" if abs(x) < 1e-9 else f"{x:.2f}".rstrip("0").rstrip(".")


np.set_printoptions(formatter={"float_kind": short})
pd.set_option("display.float_format", short)

tok = BPETokenizer.train(CORPUS)
table = hint_table(tok)
sentence = "I am hungry . I want milk ."
ids = tok.encode(sentence)
grid = table[ids]
print(sentence, "→", ids)

## 4.1 The problem: same token, same vector

After Step 2, a token's vector only depends on the token. Our sentence has `_I` twice, at positions 1 and 5 (shown with our hint rows from Step 2, which are easy to read):

| pos | token | me | be | want | stop | feel | food | hug | sleep |
|---|---|---|---|---|---|---|---|---|---|
| 1 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 5 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |

They are identical. But the first `_I` is followed by `_am`, and the second by `_want`. Step 3 measured the cost: after `_I`, the trained toddler can only say am 50%, want 50%.

A bigger problem is coming. Attention (Step 5) works by **mixing** the vectors of the text together. Mixing doesn't care about order: it only sees **which** vectors are there, like toys in a bag. Look at these two texts:

```
I want milk .    →   _I     _want  _milk  _.
milk want I .    →   _milk  _want  _I     _.
```

Both bags hold exactly the same 4 vectors: one `_I`, one `_want`, one `_milk`, one `_.`. Without positions, the model could not tell "I want milk" from "milk want I".

In [ ]:
# Try it: same token, same vector; and the same bag for two different orders
print("row 1 (_I):", grid[0])
print("row 5 (_I):", grid[4])
print("identical?", np.array_equal(grid[0], grid[4]))


def bag(text):
    """The rows of a text in sorted order: all that order-blind mixing can see."""
    return sorted(tuple(row) for row in table[tok.encode(text)])


print('\nsame bag for "I want milk ." and "milk want I ."?', bag("I want milk .") == bag("milk want I ."))

## 4.2 The idea: a vector for each position

Give every **position** its own vector, and **add** it to the token's vector:

```
input vector  =  token vector (what it is)  +  position vector (where it is)
```

Then `_I` at position 1 and `_I` at position 5 get different position vectors, so their input vectors differ.

## 4.3 The toddler's position code: a clock with one hand

Our vectors get **2 new columns** for the position, `px` and `py`. That makes **10 numbers**: 8 for meaning, 2 for position.

Picture a clock with a single hand. At position 1 it points right. Each next position turns it by **45°**, an eighth of a full turn:

- angle = (position − 1) × 45°
- `px` = how far right the hand points (the cosine of the angle)
- `py` = how far up it points (the sine of the angle)

| position | angle | px | py | the hand points |
|---|---|---|---|---|
| 1 | 0° | 1 | 0 | → |
| 2 | 45° | 0.71 | 0.71 | ↗ |
| 3 | 90° | 0 | 1 | ↑ |
| 4 | 135° | −0.71 | 0.71 | ↖ |
| 5 | 180° | −1 | 0 | ← |
| 6 | 225° | −0.71 | −0.71 | ↙ |
| 7 | 270° | 0 | −1 | ↓ |
| 8 | 315° | 0.71 | −0.71 | ↘ |

0.71 is short for 0.7071…, the cosine and sine of 45°.

![The position clock: our sentence around the clock](figures/04_position_clock.png)

The two `_I` (positions 1 and 5) now point in opposite directions.

In [ ]:
# Try it: the clock hand of each position
def position_vector(position, step_degrees=45):
    angle = np.radians((position - 1) * step_degrees)
    return np.round([np.cos(angle), np.sin(angle)], 10) + 0.0  # rounding removes float dust like 6e-17


positions = np.array([position_vector(p) for p in range(1, 9)])
pd.DataFrame(positions, index=[f"#{p}" for p in range(1, 9)], columns=["px", "py"])

## 4.4 Adding, step by step

`_I` at position 1:

```
               me   be want stop feel food  hug sleep    px    py
_I              1    0    0    0    0    0    0    0     0     0
+ position 1    0    0    0    0    0    0    0    0     1     0
= input         1    0    0    0    0    0    0    0     1     0
```

`_I` at position 5:

```
               me   be want stop feel food  hug sleep    px    py
_I              1    0    0    0    0    0    0    0     0     0
+ position 5    0    0    0    0    0    0    0    0    -1     0
= input         1    0    0    0    0    0    0    0    -1     0
```

![Adding a position to a token](figures/04_adding_positions.png)

The whole sentence, now 8 rows of 10 numbers:

| pos | token | me | be | want | stop | feel | food | hug | sleep | px | py |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 1 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 |
| 2 | `_am` | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0.71 | 0.71 |
| 3 | `_hungry` | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 0 | 1 |
| 4 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | −0.71 | 0.71 |
| 5 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | −1 | 0 |
| 6 | `_want` | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 | −0.71 | −0.71 |
| 7 | `_milk` | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | −1 |
| 8 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0.71 | −0.71 |

Rows 1 and 5 now differ, and so do rows 4 and 8 (the two `_.`). **This grid is what goes into the transformer.**

Why "add" when we only fill 2 empty columns? The token vector has 0 in the position columns, and the position vector has 0 in the meaning columns, so adding them just fills the empty columns. Real models don't keep separate columns: they add the position on top of the meaning in the **same** columns, and training learns to tell them apart. We keep them apart only so you can read the numbers.

In [ ]:
# Try it: add a position to a token, column by column
COLUMNS10 = COLUMNS + ["px", "py"]


def show_add(token, position):
    token_part = np.append(table[tok.token_to_id[token]], [0, 0])
    position_part = np.append(np.zeros(8), position_vector(position))
    print(f"{'':<14}" + "".join(f"{c:>6}" for c in COLUMNS10))
    for label, values in [(token, token_part), (f"+ position {position}", position_part),
                          ("= input", token_part + position_part)]:
        print(f"{label:<14}" + "".join(f"{short(x):>6}" for x in values))
    print()


show_add("_I", 1)
show_add("_I", 5)

In [ ]:
# Try it: the whole sentence with positions (8 rows of 10 numbers)
def with_positions(grid):
    n = len(grid)
    token_part = np.hstack([grid, np.zeros((n, 2))])
    position_part = np.hstack([np.zeros((n, grid.shape[1])), np.array([position_vector(p) for p in range(1, n + 1)])])
    return token_part + position_part


inputs = with_positions(grid)
pd.DataFrame(inputs, index=[f"{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)], columns=COLUMNS10)

## 4.5 What the dot product sees now

The dot product of two position vectors only depends on **how far apart** the two positions are:

- `#1 · #2` = 1 × 0.71 + 0 × 0.71 = **0.71** (1 step apart)
- `#4 · #5` = (−0.71) × (−1) + 0.71 × 0 = **0.71** (also 1 step apart: same answer)
- `#1 · #3` = 1 × 0 + 0 × 1 = **0** (2 steps apart)
- `#1 · #5` = 1 × (−1) + 0 × 0 = **−1** (4 steps apart: opposite sides)

| distance (steps) | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| position dot product | 1 | 0.71 | 0 | −0.71 | −1 | −0.71 | 0 | 0.71 |

This is the clever part. Attention is built on dot products (Step 5), so it will be able to tell **near** tokens from **far** ones, wherever they are in the text.

Look at the end of the table: 7 steps apart gives 0.71, like 1 step apart. On the circle, position 8 sits right next to position 1. That is the limit of a single hand (section 4.6).

The whole input vectors add up both parts. For the two `_I`:

- `_I` at #1 · `_I` at #5 = 1 (same meaning) + (−1) (opposite positions) = **0**
- `_I` at #1 · `_I` at #1 = 1 + 1 = **2**

And the bag problem is solved. In "I want milk .", `_I` is at position 1, with (px, py) = (1, 0). In "milk want I .", it is at position 3, with (0, 1). The two bags now hold different vectors.

In [ ]:
# Try it: every position dot product (the same distance always gives the same number)
labels = [f"#{p}" for p in range(1, 9)]
pd.DataFrame(positions @ positions.T, index=labels, columns=labels)

In [ ]:
# Try it: the two _I, and the bag test again, now with positions
print("_I at #1 · _I at #5 =", short(inputs[0] @ inputs[4]))
print("_I at #1 · _I at #1 =", short(inputs[0] @ inputs[0]))


def bag_with_positions(text):
    return sorted(tuple(row.round(6)) for row in with_positions(table[tok.encode(text)]))


print('\nsame bag for "I want milk ." and "milk want I ."?',
      bag_with_positions("I want milk .") == bag_with_positions("milk want I ."))

## 4.6 One hand is not enough for long texts

After 8 steps the hand has made a full turn, so position 9 would point exactly like position 1. Our sentences have 8 tokens, so one hand is enough for the toddler. Real texts are much longer.

The fix is a clock with **several hands turning at different speeds**, like the second, minute and hour hands of a real clock:

| position | fast hand (45° per step) | slow hand (5.6° per step) |
|---|---|---|
| 1 | 0° → (1, 0) | 0° → (1, 0) |
| 2 | 45° → (0.71, 0.71) | 5.6° → (1.00, 0.10) |
| 9 | 360° = 0° → (1, 0) | 45° → (0.71, 0.71) |

![A fast hand and a slow hand](figures/04_two_hands.png)

- The fast hand tells **neighbors** apart, but repeats after 8 steps.
- The slow hand tells **far** positions apart, but barely moves from one step to the next.

Together they only repeat after 64 positions. Each hand needs 2 columns.

In [ ]:
# Try it: a fast hand and a slow hand
for p in [1, 2, 9]:
    print(f"#{p}:  fast hand {position_vector(p, 45)}   slow hand {position_vector(p, 45 / 8)}")

## 4.7 How real LLMs do it

- **Sine waves, like our clock** (the original Transformer, 2017): many hands, from very fast to very slow, added to the token vectors.
- **Learned positions** (GPT-2, GPT-3): a second table with one learned row per position (1,024 rows in GPT-2, 2,048 in GPT-3), added like ours. It's simple, but the model knows nothing past its last row.
- **Rotation, "RoPE"** (Llama, Mistral, Qwen and most recent models): instead of adding a position vector, attention *turns* pairs of numbers by an angle that grows with the position. These are our clock hands again, used inside attention. The dot product then depends only on the distance between two tokens.
- **Context window:** the number of positions a model can handle. Our toddler: 8. Big models today: 128,000 tokens or more.

## 4.8 Wait: which is trained first, the embeddings or the positions?

Neither: **nothing is trained first**. It helps to separate two different orders.

**1. The order of one guess:** tokenizer → embedding → + position → attention → … → output table. This is the path the numbers take *every time the model makes a guess*.

**2. The order of building the model.** There is almost none:
- The **tokenizer** is built first, on its own, by counting (Step 1). It is not part of training.
- Then **all the number tables are created at once, all random**: the embedding table, the attention tables, the feed-forward tables, the output table.
- Our **position clock** is a fixed formula, so there is nothing in it to train. (In models with learned positions, like GPT-2, the position table also starts random and is trained along with the rest.)
- Then **training improves all the tables at the same time**, round after round, exactly as in Step 3.

**One training round, step by step:**
1. Take a text from the training data: "I am hungry . I want", right answer `_milk`.
2. Run the **whole** pipeline with the current numbers (random at first), **positions included**. Guess: milk 30%, mommy 40%, bed 30%.
3. Measure the surprise (the loss): the right answer got only 30%.
4. Backpropagation sends a **push** back through the pipeline to **every** number that took part: the output table, the feed-forward and attention tables, and the embedding rows of `_I`, `_am`, `_hungry`, `_.` and `_want`.
5. All these numbers move a tiny bit, in the same round.
6. Next text, next round, many times.

![One training round: a guess forward, pushes backward](figures/04_training_together.png)

**So how does the `_I` row get its final value?** The row is used twice in our sentence (positions 1 and 5), and in every other sentence. In each round, it receives the pushes from all its uses, added together. For example, on its first number (the pushes are made up here; Step 3 showed how to compute them):

| use of `_I` | push on its first number |
|---|---|
| at position 1 | +0.5 |
| at position 5 | −0.3 |
| **total for the `_I` row** | **+0.2** |

With a learning rate of 0.1: new = old − 0.1 × 0.2 = old − 0.02. The number started at 0.82 (its random value at birth, Step 2), so it is now **0.80**.

Because one row serves every position, the embedding can only learn **what `_I` means everywhere**. Whatever depends on the position is carried by the position vector, and the rest of the model learns to use both. That is exactly why "what" and "where" are kept as two separate vectors.

The `+` plays a part too: input = token vector + position vector, so a push on the input reaches **both** parts of the sum. Our clock is fixed and ignores its push, so the embedding row takes it. With learned positions, both tables take it.

**Think of a toddler band**: drums, guitar and a singer. Nobody learns their part alone first. They play together, badly at first; the teacher says "wrong note!"; everyone adjusts a little; they play again. After many rehearsals, every part fits the others. The embedding table, attention and output table are the musicians. The position clock is the metronome: it doesn't learn, and everyone learns to play along with it.

## 4.9 Training with positions, for real

Now we train the toddler of Step 3 again, from the same random numbers, with one change: each input vector is the token's row **followed by the clock hand of its position**, 10 numbers. The output table gets 10 columns too, so that its rows can learn to read the clock. The clock itself is never trained.

| after round | 0 | 10 | 25 | 50 | 100 | 300 |
|---|---|---|---|---|---|---|
| without positions (Step 3) | 4.13 | 2.69 | 0.95 | 0.59 | 0.53 | **0.516** |
| with positions | 4.02 | 2.53 | 0.94 | 0.42 | 0.34 | **0.321** |

![The loss with and without positions](figures/04_loss_with_positions.png)

The floor drops from 0.512 to **0.314**, and the 6 guesses after `_I` become almost sure:

| guess | without positions | with positions |
|---|---|---|
| `_I` at #1 → `_am` | 50% | **99%** |
| `_I` at #5 → `_want` | 50% | **99%** |
| `_want` at #6 → `_milk` | 33% | 33% |

**How does it do it?** Cut each score into two parts:
- the **meaning part**: the 8 numbers of `_I` · the first 8 numbers of the output row;
- the **position part**: the clock hand · the last 2 numbers of the output row.

After training, the last 2 numbers of the output rows are (1.98, 0.17) for `_am` and (−2.68, 0.06) for `_want`. The clock hand is (1, 0) at #1 and (−1, 0) at #5:

| score after `_I` | meaning part | position part | total |
|---|---|---|---|
| `_am`, at #1 | 8.47 | 1 × 1.98 + 0 × 0.17 = +1.98 | **10.45** |
| `_want`, at #1 | 8.46 | 1 × (−2.68) + 0 × 0.06 = −2.68 | 5.78 |
| `_am`, at #5 | 8.47 | (−1) × 1.98 + 0 × 0.17 = −1.98 | 6.49 |
| `_want`, at #5 | 8.46 | (−1) × (−2.68) + 0 × 0.06 = +2.68 | **11.14** |

(The parts are rounded to 2 decimals; the code below shows the exact totals.)

The meaning part is a tie: the `_I` row can't choose, because it serves both places. The position part breaks the tie. On its own, training stored "`_am` comes early, `_want` comes later" in the last 2 numbers of the output rows, so that they read our fixed clock.

`_want` at #6 → milk is still ⅓: the clock tells *where* a token is, not *what came before*. That's the job of attention.

In [ ]:
# Try it: train the toddler of Step 3 again, from the same random numbers, with the clock in every input vector
from toddler import learning
from toddler.embeddings import random_table

V = len(tok.vocab)
guesses = learning.make_guesses(tok, CORPUS)
birth_input = random_table(V, 8, seed=0)                       # the same table at birth as in Steps 2 and 3
plain_input, plain_output, plain_losses = learning.train(birth_input, random_table(V, 8, seed=1), guesses)
clock_input, clock_output, clock_losses = learning.train(birth_input, random_table(V, 10, seed=1), guesses,
                                                         use_positions=True)
for k in [0, 10, 25, 50, 100, 300]:
    print(f"after {k:>3} rounds: without positions {plain_losses[k]:.3f}   with positions {clock_losses[k]:.3f}")

In [ ]:
# Try it: the guesses after _I at #1 and #5, then the two parts of each score
I, am, want = (tok.token_to_id[t] for t in ["_I", "_am", "_want"])
for position in (1, 5):
    plain = learning.softmax(plain_output @ plain_input[I])
    clock = learning.softmax(clock_output @ learning.input_vector(clock_input, I, position, use_positions=True))
    print(f"_I at #{position}: without positions am {plain[am]:.0%}, want {plain[want]:.0%}"
          f"   with positions am {clock[am]:.0%}, want {clock[want]:.0%}")

print("\nlast 2 numbers of the output rows:  _am", clock_output[am, 8:], "  _want", clock_output[want, 8:])
for name, j in [("_am", am), ("_want", want)]:
    meaning = clock_input[I] @ clock_output[j, :8]
    for position in (1, 5):
        place = position_vector(position) @ clock_output[j, 8:]
        print(f"{name:<5} after _I at #{position}: meaning {meaning:.2f} + position {place:+.2f} = {meaning + place:.2f}")

## 4.10 What is still missing

Each row now knows **what** it is and **where** it is. But row 6 (`_want`) still knows nothing about row 3 (`_hungry`), so it can't guess `_milk`: after training, it still says ⅓ each. That is the job of **Step 5, attention**: each token looks back at the earlier ones, using dot products of meaning and of position. With it, the floor will fall again, to 0.157.

<!-- notebook-only -->

The same code is saved in `toddler/positions.py`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks gives the same inputs
from toddler import positions as toddler_positions

assert np.allclose(toddler_positions.with_positions(grid), inputs)
print("toddler/positions.py builds the same inputs")

<!-- notebook-only -->

## Playground: your own toddler world, with and without positions

Write up to 10 lines in the cell below, then run the cells of this section. The toddler trains twice, from the same random numbers: without positions (as in Step 3), then with the clock. Compare the two losses, and the probability of each right answer along one line.

Ideas to try:
- Add `"I want milk . I am hungry ."`: `_I` at #1 can now be followed by `_am` or `_want`, so even with positions the toddler can't be sure there (am 75%), and the loss can't go as low.
- Add a line of more than 8 tokens: on our one-hand clock, position 9 looks exactly like position 1.

<!-- /notebook-only -->

In [ ]:
# Playground: your own toddler world (up to 10 lines), then run this cell and the next ones
MY_CORPUS = [
    "I am hungry . I want milk .",
    "I am sad . I want mommy .",
    "I am tired . I want bed .",
    "I want milk . I am hungry .",
]
SHOW_LINE = 0      # the line to look at below (0 = the first line)
ROUNDS = 300

In [ ]:
# Playground: train without and with positions, from the same random numbers
from toddler import learning
from toddler.data import check_corpus
from toddler.embeddings import random_table

my_corpus = check_corpus(MY_CORPUS)
my_tok = BPETokenizer.train(my_corpus)
my_guesses = learning.make_guesses(my_tok, my_corpus)
my_V = len(my_tok.vocab)
longest = max(len(my_tok.encode(line)) for line in my_corpus)
if longest > 8:
    print(f"Note: a line has {longest} tokens. On our one-hand clock, position 9 looks like position 1.\n")

my_runs = {}
for label, width, use_positions in [("without positions", 8, False), ("with positions", 10, True)]:
    my_runs[label] = learning.train(random_table(my_V, 8, seed=0), random_table(my_V, width, seed=1), my_guesses,
                                    rounds=ROUNDS, use_positions=use_positions)
    losses = my_runs[label][2]
    print(f"{label:<17}: loss {losses[0]:.3f} at birth → {losses[-1]:.3f} after {ROUNDS} rounds")

In [ ]:
# Playground: the probability of each right answer along one of your lines, without and with positions
line_ids = my_tok.encode(my_corpus[SHOW_LINE])
report = {}
for position in range(1, len(line_ids)):
    seen, answer = line_ids[position - 1], line_ids[position]
    row = {}
    for label, use_positions in [("without positions", False), ("with positions", True)]:
        trained_input, trained_output, _ = my_runs[label]
        my_probs = learning.softmax(trained_output @ learning.input_vector(trained_input, seen, position, use_positions))
        row[label] = f"{my_probs[answer]:.0%}"
    report[f"#{position} {my_tok.vocab[seen]} → {my_tok.vocab[answer]}"] = row
print(my_corpus[SHOW_LINE])
pd.DataFrame(report).T

## Your turn

Do each one **by hand first**, using the tables above.

### 4.1 What is the position vector (px, py) of position 7?

<details>
<summary>Show the answer</summary>
<!-- answer: 0 -1 -->
<!-- format: two numbers, e.g. 1 0 -->

Position 7 turns the hand by (7 − 1) × 45° = 270°: straight down. So (px, py) = **(0, −1)**.

</details>

In [ ]:
my_answer = None  # two numbers, e.g. [1, 0]
check("4.1", my_answer)

### 4.2 Write the full input vector (10 numbers) of `_want` at position 6, with its hint row.

<details>
<summary>Show the answer</summary>
<!-- answer: 0 0 1 0 0 0 0 0 -0.71 -0.71 | 0 0 1 0 0 0 0 0 -0.7 -0.7 | 0 0 1 0 0 0 0 0 -0.707 -0.707 | 0 0 1 0 0 0 0 0 -0.7071 -0.7071 -->
<!-- format: 10 numbers separated by spaces -->

```
               me   be want stop feel food  hug sleep    px    py
_want           0    0    1    0    0    0    0    0     0     0
+ position 6    0    0    0    0    0    0    0    0 -0.71 -0.71
= input         0    0    1    0    0    0    0    0 -0.71 -0.71
```

Position 6 is 225°: the hand points down and to the left.

</details>

In [ ]:
my_answer = None  # 10 numbers, e.g. [0, 0, 1, 0, 0, 0, 0, 0, 0.5, 0.5]
check("4.2", my_answer)

### 4.3 Compute the position dot product of positions 2 and 4.

<details>
<summary>Show the answer</summary>
<!-- answer: 0 -->
<!-- format: one number -->

Position 2 = (0.71, 0.71) and position 4 = (−0.71, 0.71):

0.71 × (−0.71) + 0.71 × 0.71 = −0.5 + 0.5 = **0**

They are 2 steps (90°) apart, just like positions 1 and 3: same distance, same dot product.

</details>

In [ ]:
my_answer = None  # one number
check("4.3", my_answer)

<!-- notebook-only -->

### 4.4 (code) Write `my_position_vector(position)` yourself, with `math.cos` and `math.sin`.

<details>
<summary>Show the answer</summary>

```python
import math

def my_position_vector(position):
    angle = math.radians((position - 1) * 45)
    return [math.cos(angle), math.sin(angle)]
```

</details>

<!-- /notebook-only -->

In [ ]:
import math


def my_position_vector(position):
    # your code here: the hand turns 45° per position, starting at 0° for position 1
    return None


check("4.4", my_position_vector)

### 4.5 On our one-hand clock, which position looks exactly like position 1?

<details>
<summary>Show the answer</summary>
<!-- choices: *Position 9: the hand has made a full turn | Position 5: it is on the opposite side | Position 2: it is the neighbor | None: every position always looks different -->

**Position 9**: (9 − 1) × 45° = 360°, a full turn, so the hand points exactly like at position 1. That's why real models use several hands turning at different speeds.

</details>

In [ ]:
my_answer = None  # a position number
check("4.5", my_answer)

### 4.6 Think: without positions, what would attention see in "I want milk ." and "milk want I ."?

<details>
<summary>Show the answer</summary>
<!-- choices: *The same bag of 4 vectors: it could not tell the two orders apart | Two different sentences, because the words are in a different order | Only the first token of each text | Only the last token of each text -->

The same bag of 4 vectors (`_I`, `_want`, `_milk`, `_.`), because mixing ignores order. With positions, `_I` is (1, 0) in the first text and (0, 1) in the second, so the two bags differ.

</details>

### 4.7 Think: when is the embedding table trained?

<details>
<summary>Show the answer</summary>
<!-- choices: *At the same time as all the other tables, in every training round | First, before the positions are added | Last, after attention and the output table are trained | Never: its values are written by hand -->

At the same time as everything else. In every round, the whole model makes a guess (positions included), and every trained table gets its small push. Only the tokenizer is built before, by counting, and our position clock is a fixed formula that is never trained.

</details>

### 4.8 Think: trained with positions, the loss floor falls from 0.512 to 0.314. Which guesses became sure?

<details>
<summary>Show the answer</summary>
<!-- choices: *The 6 guesses after `_I`: the clock tells the first `_I` from the second | The 3 guesses after `_want`: the clock tells which thing comes next | The 3 guesses after `_am`: the clock tells which feeling comes next | None: the clock only makes training faster -->

The 6 guesses after `_I`. Without positions, each one costs −ln(1/2) = 0.69. With the clock, `_I` at #1 → `_am` and `_I` at #5 → `_want` become sure, so they cost 0. That removes 6 × 0.69 / 21 = 0.198 from the average: 0.512 − 0.198 = **0.314**. The clock can't tell which feeling or which thing comes next: that needs attention, or can never be known.

</details>

## Summary

- Same token, same vector: without help, the model can't tell *where* a token is, and attention would see the text as a bag of tokens.
- The fix: add a **position vector** to each token vector.
- The toddler's position code is a clock hand turning 45° per position, stored in 2 new columns, `px` and `py`. Vectors now have 10 numbers.
- The dot product of two position vectors depends only on their **distance**, so attention will be able to tell near from far.
- One hand repeats after 8 positions. Real models use many hands (sine waves), learned positions, or rotations (RoPE).
- Nothing is trained first: all the tables are trained at the same time, through the whole model, positions included. The clock is fixed, and the others learn to play along with it.
- Trained again with the clock, the toddler tells the two `_I` apart (am 99% at #1, want 99% at #5), and the loss floor falls from 0.512 to 0.314. After `_want`, it still says ⅓ each.
- Next: **Step 5, attention**.